In [ ]:
# Instalar biblioteca pymongo en Colab.
!pip install pymongo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 23.0 MB/s eta 0:00:00


In [ ]:
# Conexión con Secret
from google.colab import userdata
from pymongo import MongoClient

uri = userdata.get("MONGO_URI")
client = MongoClient(uri)
client.admin.command("ping")
print("Conexión creada")
print(client.list_database_names())

Conexión creada
['ecommify_test', 'sample_mflix', 'admin', 'local']


In [ ]:
# Lectura CSV Olist (para script de inserción)
!pip install -q pandas
import pandas as pd

BASE = '/content'
products_df = pd.read_csv(f'{BASE}/olist_products_dataset.csv')
items_df = pd.read_csv(f'{BASE}/olist_order_items_dataset.csv')
reviews_df = pd.read_csv(f'{BASE}/olist_order_reviews_dataset.csv')
print('CSV OK:', len(products_df), 'productos')

In [ ]:
# Inserción productos + reviews → ecommify_test
ELECTRONICS = {'eletronicos','informatica_acessorios','telefonia','pcs','eletrodomesticos'}
FASHION = {'moda_roupa_masculina','moda_roupa_feminina','bebes','beleza_saude','moda_calcados'}

def build_spec(row):
    cat = row['product_category_name'] or 'general'
    if cat in ELECTRONICS:
        return {'tipo':'electronica','peso_g':float(row['product_weight_g']) if pd.notna(row['product_weight_g']) else None,
                'dimensiones_cm':{'largo':float(row['product_length_cm']) if pd.notna(row['product_length_cm']) else None,
                                  'alto':float(row['product_height_cm']) if pd.notna(row['product_height_cm']) else None,
                                  'ancho':float(row['product_width_cm']) if pd.notna(row['product_width_cm']) else None}}
    if cat in FASHION:
        return {'tipo':'moda_belleza','peso_g':float(row['product_weight_g']) if pd.notna(row['product_weight_g']) else None,
                'material':'textil','talla_estandar':'M'}
    return {'tipo':'general','peso_g':float(row['product_weight_g']) if pd.notna(row['product_weight_g']) else None}

sales = items_df.groupby('product_id').agg(total_units_sold=('order_item_id','count'), price_avg=('price','mean')).reset_index()
op = items_df[['order_id','product_id']].drop_duplicates()
orv = reviews_df.merge(op, on='order_id')
ratings = orv.groupby('product_id')['review_score'].mean().reset_index(name='average_rating')
merged = products_df.merge(sales, on='product_id').merge(ratings, on='product_id', how='left')
merged = merged.sort_values('total_units_sold', ascending=False).head(1200)

db = client['ecommify_test']
pcol, rcol = db['productos'], db['reviews']
pcol.delete_many({}); rcol.delete_many({})

docs = []
for _, row in merged.iterrows():
    avg = row['average_rating']
    docs.append({
        'product_id': row['product_id'],
        'name': f"Producto Ecommify {row['product_id'][:8]}",
        'category': row['product_category_name'] or 'sin_categoria',
        'price': round(float(row['price_avg']),2),
        'specifications': build_spec(row),
        'computed_metrics': {
            'total_units_sold': int(row['total_units_sold']),
            'average_rating': round(float(avg),2) if pd.notna(avg) else None
        }
    })
pcol.insert_many(docs)

pids = set(merged['product_id'])
revs = []
for _, r in orv[orv['product_id'].isin(pids)].head(5000).iterrows():
    revs.append({'review_id':r['review_id'],'product_id':r['product_id'],'order_id':r['order_id'],
                 'score':int(r['review_score']),'comment':str(r['review_comment_message']) if pd.notna(r['review_comment_message']) else ''})
rcol.insert_many(revs)
print('Productos:', pcol.count_documents({}))
print('Reviews:', rcol.count_documents({}))

In [ ]:
# Inserción geolocation → ecommify_test
!pip install -q pandas pymongo
import pandas as pd
from pymongo import GEOSPHERE

BASE = '/content'
db = client['ecommify_test']

geo_df = pd.read_csv(f'{BASE}/olist_geolocation_dataset.csv')
agg = (
    geo_df.groupby('geolocation_zip_code_prefix', as_index=False)
    .agg(
        geolocation_lat=('geolocation_lat', 'mean'),
        geolocation_lng=('geolocation_lng', 'mean'),
        geolocation_city=('geolocation_city', 'first'),
        geolocation_state=('geolocation_state', 'first'),
        record_count=('geolocation_lat', 'count'),
    )
)

gcol = db['geolocation']
gcol.delete_many({})

geo_docs = []
for _, row in agg.iterrows():
    geo_docs.append({
        'geolocation_zip_code_prefix': str(int(row['geolocation_zip_code_prefix'])).zfill(5),
        'location': {
            'type': 'Point',
            'coordinates': [float(row['geolocation_lng']), float(row['geolocation_lat'])],
        },
        'geolocation_city': str(row['geolocation_city']),
        'geolocation_state': str(row['geolocation_state']),
        'metadata': {'record_count': int(row['record_count'])},
    })

gcol.insert_many(geo_docs, ordered=False)
gcol.create_index('geolocation_zip_code_prefix', unique=True)
gcol.create_index([('location', GEOSPHERE)])
print('Geolocation:', gcol.count_documents({}))

Geolocation: 19015


In [ ]:
# Conexión a MongoDB Atlas y verificación de datos en productos y reviews
!pip install -q pymongo

from google.colab import userdata
from pymongo import MongoClient

uri = userdata.get("MONGO_URI")
client = MongoClient(uri)
print("Conexión OK")

db = client["ecommify_test"]
productos = db["productos"]
reviews = db["reviews"]

print("Productos:", productos.count_documents({}))
print("Reviews:", reviews.count_documents({}))
print("\nEjemplo producto:")
print(productos.find_one())

Conexión OK
Productos: 1201
Reviews: 5000

Ejemplo producto:
{'_id': ObjectId('6a1e3e6c6a686960113990fb'), 'product_id': '154e7e31ebfa092203795c972e5804a6', 'name': 'Producto Ecommify 154e7e31', 'category': 'beleza_saude', 'price': 22.51, 'specifications': {'tipo': 'moda_belleza', 'peso_g': 100.0, 'material': 'textil', 'talla_estandar': 'M'}, 'computed_metrics': {'total_units_sold': 281, 'average_rating': 4.32}}


In [ ]:
# Agregación: promedio de rating por categoría (top 5)
pipeline = [
    {"$match": {"computed_metrics.average_rating": {"$ne": None}}},
    {"$group": {"_id": "$category", "promedio_rating": {"$avg": "$computed_metrics.average_rating"}}},
    {"$sort": {"promedio_rating": -1}},
    {"$limit": 5},
]
for row in productos.aggregate(pipeline):
    print(row)

In [ ]:
# Reviews de un producto usando product_id como referencia
ej = productos.find_one()
pid = ej["product_id"]
print("Producto:", pid)
for r in reviews.find({"product_id": pid}).limit(5):
    print("Score:", r["score"], "| review_id:", r["review_id"][:8])

Producto: 154e7e31ebfa092203795c972e5804a6
Score: 1 | review_id: 84ae94ac
Score: 4 | review_id: 785eeef6
Score: 4 | review_id: 00a1b1c8
Score: 5 | review_id: 445853ac
Score: 3 | review_id: 23f1e050


In [ ]:
# U5-1: Medir rendimiento ANTES de crear índices (baseline con explain)

import time

db = client['ecommify_test']
productos = db['productos']
reviews = db['reviews']

filtro = {
    'category': 'cama_mesa_banho',
    'computed_metrics.average_rating': {'$gte': 4},
}

plan_antes = db.command(
    'explain',
    {
        'find': 'productos',
        'filter': filtro,
        'sort': {'computed_metrics.total_units_sold': -1},
        'limit': 20,
    },
    verbosity='executionStats',
)

print('ANTES executionTimeMillis:', plan_antes['executionStats']['executionTimeMillis'])
print('ANTES stage:', plan_antes['executionStats']['executionStages'].get('stage'))

ANTES executionTimeMillis: 0
ANTES stage: SORT


In [ ]:
# U5-2: Crear índices compuesto, parcial, texto y en reviews

productos.create_index([
    ('category', 1),
    ('computed_metrics.average_rating', -1),
])

productos.create_index(
    [('computed_metrics.total_units_sold', -1)],
    partialFilterExpression={'computed_metrics.average_rating': {'$gte': 4}},
    name='idx_top_sellers_rated',
)

productos.create_index([
    ('name', 'text'),
    ('category', 'text'),
])

reviews.create_index([
    ('product_id', 1),
    ('score', -1),
])

print('Índices productos:', list(productos.index_information().keys()))
print('Índices reviews:', list(reviews.index_information().keys()))

Índices productos: ['_id_', 'product_id_1', 'category_1', 'category_1_computed_metrics.average_rating_-1', 'idx_top_sellers_rated', 'name_text_category_text']
Índices reviews: ['_id_', 'product_id_1', 'product_id_1_score_-1']


In [ ]:
# U5-3: Medir rendimiento DESPUÉS de índices (misma consulta que U5-1)

plan_despues = db.command(
    'explain',
    {
        'find': 'productos',
        'filter': filtro,
        'sort': {'computed_metrics.total_units_sold': -1},
        'limit': 20,
    },
    verbosity='executionStats',
)

print('DESPUÉS executionTimeMillis:', plan_despues['executionStats']['executionTimeMillis'])
print('DESPUÉS stage:', plan_despues['executionStats']['executionStages'].get('stage'))
print('DESPUÉS indexName:', plan_despues['executionStats']['executionStages'].get('indexName'))

DESPUÉS executionTimeMillis: 2
DESPUÉS stage: SORT
DESPUÉS indexName: None


In [ ]:
# U5-4: Pipeline optimizado con 5+ stages ($match, $lookup, $unwind, $group, $addFields, $sort, $limit)

t0 = time.perf_counter()

pipeline_u5 = [
    {'$match': {'computed_metrics.average_rating': {'$gte': 4}}},
    {'$lookup': {
        'from': 'reviews',
        'localField': 'product_id',
        'foreignField': 'product_id',
        'as': 'product_reviews',
    }},
    {'$unwind': '$product_reviews'},
    {'$group': {
        '_id': '$category',
        'promedio_rating': {'$avg': '$computed_metrics.average_rating'},
        'total_reviews': {'$sum': 1},
        'ventas_promedio': {'$avg': '$computed_metrics.total_units_sold'},
    }},
    {'$addFields': {
        'score_compuesto': {'$multiply': ['$promedio_rating', '$ventas_promedio']},
    }},
    {'$sort': {'score_compuesto': -1}},
    {'$limit': 10},
]

result = list(productos.aggregate(pipeline_u5, allowDiskUse=True))

print(f'Tiempo pipeline: {(time.perf_counter() - t0) * 1000:.2f} ms')
for row in result[:5]:
    print(row)

Tiempo pipeline: 98.96 ms
{'_id': 'ferramentas_jardim', 'promedio_rating': 4.1721097046413504, 'total_reviews': 237, 'ventas_promedio': 269.9957805907173, 'score_compuesto': 1126.4520164147484}
{'_id': 'moveis_decoracao', 'promedio_rating': 4.185444444444444, 'total_reviews': 180, 'ventas_promedio': 205.72222222222223, 'score_compuesto': 861.0389320987655}
{'_id': 'relogios_presentes', 'promedio_rating': 4.1984976525821605, 'total_reviews': 213, 'ventas_promedio': 125.2018779342723, 'score_compuesto': 525.6597906059204}
{'_id': 'informatica_acessorios', 'promedio_rating': 4.281592592592593, 'total_reviews': 270, 'ventas_promedio': 118.91851851851852, 'score_compuesto': 509.16064801097394}
{'_id': 'beleza_saude', 'promedio_rating': 4.263590909090909, 'total_reviews': 440, 'ventas_promedio': 103.07045454545455, 'score_compuesto': 439.4502529958678}


In [ ]:
# U5-5: Probar búsqueda full-text con el índice de texto

for doc in productos.find({'$text': {'$search': 'Ecommify'}}).limit(3):
    print(doc['name'], '|', doc['category'])

Producto Ecommify 8f83335d | perfumaria
Producto Ecommify 9af7b4a3 | market_place
Producto Ecommify 90ef6790 | papelaria
